# Tutorial 9 — Reinforcement Learning on Discrete Geometric Structures

**MM845 — Tópicos de Geometria III: AI for Geometry**
Paired with **Lecture 9: Introduction to Reinforcement Learning**

---

Every tutorial so far started from a dataset. Reinforcement learning does not have
one. There is a **state space**, a set of **actions** that move between states, and a
**reward**; the agent generates its own data by acting, and must discover a policy
$\pi : S \to A$ that maximises expected discounted return.

For a geometer the appeal is that many natural problems already have this shape.
A triangulation with Pachner moves, a knot diagram with Reidemeister moves, a
polytope with bistellar flips — these are state spaces with a group or groupoid of
legal transformations, and asking for an optimal sequence of moves is asking for a
policy. Wagner's counterexample search (Lecture 1) is exactly this idea used
seriously.

We do two problems, chosen so that both have **exactly checkable answers**.

| § | Question |
|---|---|
| 1 | An MDP on a triangulated surface; shortest paths |
| 2 | Tabular Q-learning, checked against Dijkstra |
| 3 | Stability: what $\varepsilon$, $\alpha$ and $\gamma$ actually do |
| 4 | A combinatorial MDP: edge flips, and rediscovering Delaunay |
| 5 | Summary |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import Delaunay
import heapq

SEED = 20260925
rng = np.random.default_rng(SEED)

GEO_DARK, GEO_TEAL, GEO_RUST = "#103158", "#006c86", "#b2461e"
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.titlesize": 10,
                     "axes.grid": True, "grid.alpha": 0.25,
                     "axes.prop_cycle": plt.cycler(color=[GEO_DARK, GEO_TEAL, GEO_RUST])})
print("numpy", np.__version__)

---
## 1. A Markov decision process on a triangulation

Take a triangulated region of the plane. The **states** are its vertices; from a
vertex the **actions** are the incident edges; the **transition** is deterministic —
traverse the edge. Let the reward for traversing an edge be minus its length, plus a
bonus on reaching a designated target.

Maximising undiscounted return is then exactly minimising total length: the optimal
policy is a **shortest path**, which on a fine triangulation of a surface approximates
a geodesic. That is the point of choosing this problem — Dijkstra gives us the exact
answer, so we can grade the learner rather than admire it.

Two remarks on the formalism, both worth stating precisely.

- The problem is Markov because the optimal continuation depends only on the current
  vertex, not on how you arrived. That is what makes a value function well defined.
- The optimal value $V^\star(s)$ (the negative of the distance-to-target) satisfies
  the **Bellman equation** $V^\star(s) = \max_a [r(s,a) + \gamma V^\star(s')]$.
  Dijkstra solves this directly because it knows the transition model. Q-learning does
  not know the model, and must estimate it from experience — which is the entire
  difference between planning and learning.

In [ ]:
N_V = 90
pts = rng.uniform(0, 1, size=(N_V, 2))
pts = np.vstack([pts, [[0, 0], [1, 0], [0, 1], [1, 1]]])       # pin the corners
tri = Delaunay(pts)

edges = set()
for s in tri.simplices:
    for a, b in ((0, 1), (1, 2), (2, 0)):
        i, j = sorted((int(s[a]), int(s[b])))
        edges.add((i, j))
edges = sorted(edges)

n = len(pts)
neighbours = [[] for _ in range(n)]
for i, j in edges:
    w = float(np.linalg.norm(pts[i] - pts[j]))
    neighbours[i].append((j, w))
    neighbours[j].append((i, w))

START, GOAL = int(np.argmin((pts**2).sum(1))), int(np.argmin(((pts - 1)**2).sum(1)))
print(f"{n} vertices, {len(edges)} edges;  start {START}  goal {GOAL}")
print(f"max degree {max(len(nb) for nb in neighbours)}")


def dijkstra(neighbours, goal):
    """Exact distance-to-goal for every vertex."""
    dist = {goal: 0.0}
    pq = [(0.0, goal)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist.get(u, np.inf):
            continue
        for v, w in neighbours[u]:
            nd = d + w
            if nd < dist.get(v, np.inf):
                dist[v] = nd
                heapq.heappush(pq, (nd, v))
    return np.array([dist.get(i, np.inf) for i in range(len(neighbours))])


d_exact = dijkstra(neighbours, GOAL)
print(f"exact shortest-path length from start to goal: {d_exact[START]:.4f}")
print(f"straight-line distance (a lower bound):        "
      f"{np.linalg.norm(pts[START] - pts[GOAL]):.4f}")

---
## 2. Tabular Q-learning, graded against the exact answer

Q-learning estimates the action-value $Q(s,a)$ — the return from taking $a$ in $s$
and behaving optimally afterwards — by the update

$$Q(s,a) \;\leftarrow\; Q(s,a) \;+\; \alpha\Big[\, r + \gamma \max_{a'} Q(s',a') \;-\; Q(s,a) \,\Big] .$$

The bracket is the **temporal-difference error**: the gap between the current
estimate and a one-step-better estimate. Note what makes this work — the target
$r + \gamma\max_{a'}Q(s',a')$ uses the *greedy* action even when the agent behaved
otherwise. That is why Q-learning is **off-policy**, and why it can explore
randomly while still learning the optimal policy.

Exploration is by $\varepsilon$-greedy: act randomly with probability $\varepsilon$,
greedily otherwise. Since our graph is undirected and connected, sufficient random
walking visits every state, which is what the convergence theorem requires.

In [ ]:
def q_learning(neighbours, start, goal, episodes=4000, alpha=0.5, gamma=1.0,
               eps0=1.0, eps_min=0.05, max_steps=300, rng=None, goal_bonus=5.0):
    """Tabular Q-learning. Q[s] is an array over the actions available at s."""
    Q = [np.zeros(len(nb)) for nb in neighbours]
    returns = []
    for ep in range(episodes):
        eps = max(eps_min, eps0 * (1 - ep / (0.8 * episodes)))
        s, total = start, 0.0
        for _ in range(max_steps):
            if rng.random() < eps:
                a = rng.integers(len(neighbours[s]))
            else:
                a = int(np.argmax(Q[s]))
            s2, w = neighbours[s][a]
            r = -w + (goal_bonus if s2 == goal else 0.0)
            total += r
            best_next = 0.0 if s2 == goal else float(np.max(Q[s2]))
            Q[s][a] += alpha * (r + gamma * best_next - Q[s][a])
            s = s2
            if s == goal:
                break
        returns.append(total)
    return Q, np.array(returns)


def greedy_path(Q, neighbours, start, goal, max_steps=300):
    path, s = [start], start
    for _ in range(max_steps):
        if s == goal:
            break
        s = neighbours[s][int(np.argmax(Q[s]))][0]
        if s in path:                       # a cycle: the policy is not yet valid
            path.append(s)
            return path, False
        path.append(s)
    return path, s == goal


GOAL_BONUS = 5.0
Q, returns = q_learning(neighbours, START, GOAL, rng=np.random.default_rng(0), goal_bonus=GOAL_BONUS)
path, ok = greedy_path(Q, neighbours, START, GOAL)
length = sum(np.linalg.norm(pts[a] - pts[b]) for a, b in zip(path, path[1:]))

print(f"greedy policy reaches the goal: {ok}")
print(f"  learned path length {length:.4f}   in {len(path)-1} steps")
print(f"  exact optimum       {d_exact[START]:.4f}")
print(f"  excess              {100*(length/d_exact[START] - 1):.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11.4, 3.4))

ax = axes[0]
for i, j in edges:
    ax.plot(pts[[i, j], 0], pts[[i, j], 1], color="0.85", lw=0.6, zorder=1)
ax.plot(pts[path, 0], pts[path, 1], color=GEO_RUST, lw=2.2, zorder=3, label="Q-learning")
d_start = dijkstra(neighbours, START)
opt, s_ = [GOAL], GOAL
while s_ != START:                                   # walk the exact optimum back
    s_ = min(neighbours[s_], key=lambda t: d_start[t[0]] + t[1])[0]
    opt.append(s_)
ax.plot(pts[opt, 0], pts[opt, 1], color=GEO_TEAL, lw=1.2, ls="--", zorder=4, label="Dijkstra")
ax.scatter(*pts[[START, GOAL]].T, s=70, color=GEO_DARK, zorder=5)
ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
ax.legend(fontsize=7.5); ax.set_title("learned path vs exact optimum")

ax = axes[1]
w = 100
ax.plot(np.convolve(returns, np.ones(w) / w, "valid"), color=GEO_DARK)
ax.axhline(-d_exact[START] + 5.0, color=GEO_TEAL, ls="--", lw=1.2, label="optimal return")
ax.set_xlabel("episode"); ax.set_ylabel("return (moving average)")
ax.set_title("learning curve"); ax.legend(fontsize=8)

ax = axes[2]
V = np.array([np.max(q) if len(q) else 0.0 for q in Q])
finite = np.isfinite(d_exact)
vis = finite & (V != 0)
ax.scatter(-d_exact[finite & ~vis], V[finite & ~vis], s=12, color="0.75", alpha=0.8,
           label="never updated (the goal state itself)")
ax.scatter(-d_exact[vis], V[vis], s=12, color=GEO_RUST, alpha=0.7, label="visited")
lims = np.array([(-d_exact[finite]).min(), 0])
ax.plot(lims, lims + GOAL_BONUS, "k--", lw=1, label=f"exact: $y=x+{GOAL_BONUS:.0f}$")
ax.legend(fontsize=7)
ax.set_xlabel(r"exact $-\,$dist to goal"); ax.set_ylabel(r"learned $\max_a Q(s,a)$")
ax.set_title("the value function, learned vs true")
plt.tight_layout(); plt.show()

visited = finite & (V != 0)
print(f"states with any learned value : {visited.sum()} of {finite.sum()}")
print(f"correlation, all states       : {np.corrcoef(-d_exact[finite], V[finite])[0,1]:.3f}")
print(f"correlation, visited states   : {np.corrcoef(-d_exact[visited], V[visited])[0,1]:.3f}")

The greedy policy is **exactly optimal** — it recovers Dijkstra's path length to
four decimal places.

The value-function panel repays a careful look, and the two printed correlations are
the reason. On the states the agent actually updated, the learned value matches
$-\,\mathrm{dist} + \text{goal bonus}$ with correlation $1.000$ and essentially zero
residual — Q-learning has solved the (bonus-shifted) Bellman equation there, exactly.
The rust points do **not** sit on the plain identity line $y=x$; they sit on the
dashed line $y = x + 5$, because every episode's return includes the one-time
`goal_bonus` collected on the final transition into the goal. Over *all* states the
correlation collapses to about $0.36$ (excluding just that one never-touched point
recovers a correlation of exactly $1.000$ over the rest).

The gap is caused by a **single vertex — the goal itself**. Its $Q$ row is never
updated, because an episode ends the instant the agent arrives there (`if s == goal:
break`), so there is no outgoing transition from the goal in this run to generate a
TD update. Its true distance-to-goal is trivially $0$, and its untouched $Q$-row is
still $0$ from initialisation — so this one point happens to land exactly on
$y=x$, not because Q-learning learned anything correct about it, but because both of
its coordinates are trivially zero. That single point, plotted in grey, is enough to
wreck a correlation computed over ninety-odd points.

Two lessons, and they are independent of each other.

- **Q-learning is not a solver for the whole Bellman equation.** It improves
  estimates along the trajectories its own policy generates, and says nothing about
  states it never reaches — including, structurally, the goal state's own row.
  Dijkstra computes distance-to-goal for *every* vertex in $O(E\log V)$ because it
  exploits the model. Q-learning learns a good policy from the start state while
  remaining ignorant of parts of the graph — a liability if you wanted a full value
  map, and exactly the right trade when the state space cannot be enumerated at all.
  §4 is that case.
- **A single outlier moved a correlation from $1.00$ to $0.36$.** Tutorial 8 made the
  complementary point — a correlation of $0.974$ that concealed real structure. Taken
  together: a correlation coefficient is a one-number summary of a scatter plot, and
  you should look at the scatter plot — and check what the reference line in that
  scatter plot actually represents, since here it is shifted by a reward bonus that is
  easy to forget about.

> **Exercise 1 — planning versus learning.**
> (a) Dijkstra needs $O(E\log V)$ and gets the exact answer; Q-learning needed
> thousands of episodes for an approximation. Given that, when is RL the right tool?
> *enumerate*, and consider what happens when the state space is the set of
> triangulations of a surface.
>
> (b) Make the environment stochastic: with probability $p$ the agent slips to a
> random neighbour instead of its chosen one. Dijkstra on the mean graph no longer
> gives the optimal policy — explain why, and check numerically for $p = 0.2$.
>
> (c) Put the triangulation on a curved surface (lift the points to a graph
> $z = f(x,y)$ and use 3-D edge lengths). The learned path now approximates a
> **geodesic**. Compare with the straight line in the parameter domain.
> random neighbour instead of its chosen one. Dijkstra on the mean graph no longer> gives the optimal policy — explain why, and check numerically for $p = 0.2$.

---
## 3. Stability: the three knobs

Lecture 9 warned that RL is less stable than supervised learning. Here that is
concrete and cheap to demonstrate: sweep each hyperparameter and watch the quality of
the resulting policy.

- $\varepsilon$ controls exploration. Too little and the agent never discovers a
  route; too much and it never exploits what it knows.
- $\alpha$ is the learning rate of the TD update. Too large and the estimates
  oscillate; too small and they crawl.
- $\gamma$ discounts the future. It is not merely a tuning constant: it determines
  *which problem you are solving*. Small $\gamma$ makes distant rewards nearly
  invisible, so the optimal policy for the discounted problem may not be the shortest
  path at all.

In [ ]:
def evaluate_policy(Q):
    p, ok = greedy_path(Q, neighbours, START, GOAL)
    if not ok:
        return np.nan
    return sum(np.linalg.norm(pts[a] - pts[b]) for a, b in zip(p, p[1:]))


sweeps = {
    "$\\varepsilon_{\\min}$": ("eps_min", [0.0, 0.02, 0.05, 0.2, 0.5]),
    "$\\alpha$": ("alpha", [0.05, 0.2, 0.5, 0.9]),
    "$\\gamma$": ("gamma", [0.7, 0.9, 0.97, 0.99, 1.0]),
}
results = {}
for label, (key, vals) in sweeps.items():
    row = []
    for v in vals:
        lens = []
        for seed in range(3):
            Qs, _ = q_learning(neighbours, START, GOAL, episodes=2500,
                               rng=np.random.default_rng(seed), **{key: v})
            lens.append(evaluate_policy(Qs))
        row.append((v, np.nanmean(lens), np.mean(np.isnan(lens))))
    results[label] = row
    print(f"{label}")
    for v, m, fail in row:
        got = "no valid policy" if np.isnan(m) else f"length {m:.4f}  ({100*(m/d_exact[START]-1):+.1f}%)"
        print(f"    {v:<6} {got}    failures {fail:.0%}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11.2, 3.1))
for ax, (label, row) in zip(axes, results.items()):
    xs = [r[0] for r in row]
    ys = [r[1] / d_exact[START] for r in row]
    ax.plot(xs, ys, "o-", color=GEO_DARK)
    ax.axhline(1.0, color=GEO_TEAL, ls="--", lw=1.2, label="optimal")
    ax.set_xlabel(label); ax.set_ylabel("path length / optimum")
    ax.set_title(f"sweep over {label}")
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

Three different behaviours, and the differences are informative.

**$\varepsilon$ barely matters here.** Every setting from pure greedy to half-random
finds the optimum. That is a property of *this* problem rather than a general fact:
the graph is small and well connected, the reward is dense (every step is scored),
and even a greedy policy on a near-zero initial table wanders enough to explore. On
a problem with sparse rewards — a bonus only at a distant goal — exploration becomes
the binding constraint and this panel would look completely different.

**$\alpha$ has a clear floor.** At $\alpha = 0.05$ the estimates simply have not
converged within the episode budget; from $0.2$ upward it is fine. This is the
familiar step-size trade-off, and note that here larger is better because the
environment is deterministic — with stochastic transitions, a large $\alpha$ would
make the estimates chase noise.

**$\gamma$ is not a tuning constant at all.** As $\gamma \to 1$ the agent solves the
problem we actually posed; at small $\gamma$ it optimises something else — a
short-sighted objective in which reaching the goal later is worth almost nothing.
A poor result there is not a failure of learning but the correct answer to a
different question, which is a distinction worth keeping in view whenever an RL
result disappoints.

---
## 4. A combinatorial MDP: flipping edges towards Delaunay

Now a state space that is genuinely combinatorial, and a classical theorem to check
against.

Given a triangulation of a fixed planar point set, an **edge flip** replaces the
diagonal of a convex quadrilateral formed by two adjacent triangles with the other
diagonal. Flips connect the space of all triangulations of a point set, and the
classical result is:

> repeatedly flipping any edge that fails the local Delaunay (empty-circumcircle)
> test terminates, and terminates at the **Delaunay triangulation** — the unique
> triangulation maximising the minimum angle.

So we have a combinatorial state space, a natural action set, and a known optimum.
We let Q-learning search it with reward $= $ improvement in the minimum angle, and
compare with what `scipy.spatial.Delaunay` returns.

In [ ]:
def tri_min_angle(P, T):
    """Smallest angle over all triangles, in degrees."""
    best = 180.0
    for t in T:
        A, B, C = P[t[0]], P[t[1]], P[t[2]]
        for (u, v, w) in ((A, B, C), (B, C, A), (C, A, B)):
            e1, e2 = v - u, w - u
            c = np.clip(e1 @ e2 / (np.linalg.norm(e1) * np.linalg.norm(e2) + 1e-15), -1, 1)
            best = min(best, np.degrees(np.arccos(c)))
    return best


def flippable(T):
    """List of (t1, t2, shared edge, the two opposite vertices) for adjacent pairs."""
    edge_map = {}
    for ti, t in enumerate(T):
        for a, b in ((0, 1), (1, 2), (2, 0)):
            e = tuple(sorted((t[a], t[b])))
            edge_map.setdefault(e, []).append(ti)
    out = []
    for e, ts in edge_map.items():
        if len(ts) == 2:
            t1, t2 = ts
            o1 = [v for v in T[t1] if v not in e][0]
            o2 = [v for v in T[t2] if v not in e][0]
            out.append((t1, t2, e, o1, o2))
    return out


def is_convex_quad(P, e, o1, o2):
    """The flip is legal only if the quadrilateral o1-e0-o2-e1 is convex."""
    quad = [o1, e[0], o2, e[1]]
    s = []
    for i in range(4):
        a, b, c = P[quad[i]], P[quad[(i + 1) % 4]], P[quad[(i + 2) % 4]]
        d1, d2 = b - a, c - b
        cross_z = d1[0] * d2[1] - d1[1] * d2[0]     # 2-D cross product (np.cross needs 3-vectors in NumPy 2.x)
        s.append(np.sign(cross_z))
    return abs(sum(s)) == 4


def apply_flip(T, t1, t2, e, o1, o2):
    T = [list(t) for i, t in enumerate(T) if i not in (t1, t2)]
    T.append([o1, o2, e[0]])
    T.append([o1, o2, e[1]])
    return [tuple(sorted(t)) for t in T]


def state_key(T):
    return tuple(sorted(tuple(sorted(t)) for t in T))

In [ ]:
P = rng.uniform(0, 1, size=(11, 2))
D_opt = Delaunay(P)
T_opt = [tuple(sorted(map(int, s))) for s in D_opt.simplices]
best_angle = tri_min_angle(P, T_opt)

# a deliberately poor starting triangulation: flip away from Delaunay at random
T0 = list(T_opt)
for _ in range(40):
    cand = [f for f in flippable(T0) if is_convex_quad(P, f[2], f[3], f[4])]
    if not cand:
        break
    f = cand[rng.integers(len(cand))]
    T1 = apply_flip(T0, *f)
    if len(T1) == len(T0):
        T0 = T1

print(f"{len(P)} points, {len(T_opt)} triangles")
print(f"min angle — Delaunay optimum : {best_angle:.3f} deg")
print(f"min angle — scrambled start  : {tri_min_angle(P, T0):.3f} deg")

In [ ]:
def q_learn_flips(P, T_start, episodes=400, steps=25, alpha=0.5, gamma=0.9,
                  eps0=1.0, eps_min=0.1, rng=None):
    """Q-learning over triangulations; states are keyed by their triangle set."""
    Q = {}
    best_seen, best_T = -np.inf, None
    hist = []
    for ep in range(episodes):
        eps = max(eps_min, eps0 * (1 - ep / (0.8 * episodes)))
        T = list(T_start)
        ang = tri_min_angle(P, T)
        for _ in range(steps):
            k = state_key(T)
            acts = [f for f in flippable(T) if is_convex_quad(P, f[2], f[3], f[4])]
            if not acts:
                break
            if k not in Q:
                Q[k] = np.zeros(len(acts))
            if len(Q[k]) != len(acts):
                Q[k] = np.zeros(len(acts))
            a = int(rng.integers(len(acts))) if rng.random() < eps else int(np.argmax(Q[k]))
            T2 = apply_flip(T, *acts[a])
            if len(T2) != len(T):
                break
            ang2 = tri_min_angle(P, T2)
            r = ang2 - ang                                  # reward = improvement
            k2 = state_key(T2)
            acts2 = [f for f in flippable(T2) if is_convex_quad(P, f[2], f[3], f[4])]
            if k2 not in Q or len(Q[k2]) != len(acts2):
                Q[k2] = np.zeros(len(acts2))
            nxt = float(np.max(Q[k2])) if len(Q[k2]) else 0.0
            Q[k][a] += alpha * (r + gamma * nxt - Q[k][a])
            T, ang = T2, ang2
            if ang > best_seen:
                best_seen, best_T = ang, list(T)
        hist.append(best_seen)
    return Q, best_seen, best_T, np.array(hist)


Q_f, best_found, T_found, hist = q_learn_flips(P, T0, rng=np.random.default_rng(1))
print(f"best min-angle found by Q-learning : {best_found:.3f} deg")
print(f"Delaunay optimum                   : {best_angle:.3f} deg")
print(f"reached the optimum: {abs(best_found - best_angle) < 1e-6}")
print(f"states visited: {len(Q_f)}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11.2, 3.3))
for ax, (T, name) in zip(axes[:2], [(T0, f"start: {tri_min_angle(P, T0):.1f}$^\\circ$"),
                                    (T_found, f"Q-learning: {best_found:.1f}$^\\circ$")]):
    for t in T:
        for a, b in ((0, 1), (1, 2), (2, 0)):
            ax.plot(P[[t[a], t[b]], 0], P[[t[a], t[b]], 1], color=GEO_DARK, lw=1.0)
    ax.scatter(*P.T, s=22, color=GEO_RUST, zorder=4)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([]); ax.set_title(name)

axes[2].plot(hist, color=GEO_DARK)
axes[2].axhline(best_angle, color=GEO_TEAL, ls="--", lw=1.3, label="Delaunay optimum")
axes[2].set_xlabel("episode"); axes[2].set_ylabel("best min angle found (deg)")
axes[2].legend(fontsize=8); axes[2].set_title("search progress")
plt.tight_layout(); plt.show()

The agent recovers the Delaunay triangulation — the global optimum of the minimum
angle — by searching a combinatorial space with no knowledge of the empty-circumcircle
criterion, guided only by a scalar reward.

It is worth being precise about what that does and does not demonstrate. The flip
algorithm solves this problem in polynomial time and is *provably* correct; Q-learning
is far slower and provably nothing. What the experiment shows is that the **framing**
works: a space of combinatorial structures, a set of legal moves, and a quantity to
maximise is enough to define a learnable problem. That framing is the transferable
part, because it applies verbatim to problems where no flip theorem exists — searching
triangulations of a $3$-manifold for a small one, or knot diagrams for a short
unknotting sequence, or graphs for a counterexample.

Note also the practical obstacle, visible in the printed state count: we tabulated
$Q$ over triangulations, and the number of triangulations grows exponentially in the
number of points. Tabular methods die immediately at any interesting size, which is
precisely why one replaces the table with a **function approximator** — a network on
a graph representation of the state. That is Lecture 10's subject, and the reason
geometric deep learning and RL are usually deployed together.

> **Exercise 2 — a harder combinatorial objective.**
> (a) Change the reward to $-\,$(total edge length). The minimiser is the *minimum
> weight triangulation*, which — unlike Delaunay — is not achieved by greedy flipping.
> Does Q-learning beat a greedy flip baseline?
>
> (b) Count states as you increase the number of points from 8 to 14 and plot the
> growth. Estimate at what size a tabular method becomes hopeless.
>
> (c) Replace the table by a small function approximator: featurise a triangulation
> by (min angle, mean angle, total length, number of obtuse triangles) and fit a
> linear $Q$. Does it generalise across states it has never visited?

---
## 5. What to take away

- **RL needs no dataset**, only a state space, legal moves and a reward. A great deal
  of discrete geometry already has that structure.
- **Q-learning is off-policy**: it learns the value of behaving optimally while
  behaving randomly, which is what lets exploration and optimisation coexist.
- **Check against exact answers when you can.** We had Dijkstra for the paths and
  the Delaunay theorem for the flips; both turned a plausible-looking result into a
  measured one.
- **$\gamma$ chooses the problem, not just the tuning.** A disappointing RL result is
  often the correct solution to a different objective.
- **Tabular methods die with the state space.** The exponential growth of
  triangulations is exactly why function approximation — and, for geometric states,
  *geometric* function approximation — is the next step.

### Next

**Lecture 10** builds symmetry into the architecture in general: group actions,
equivariance, and graph neural networks. **Tutorial 10** compares invariant and
non-invariant models directly, which is also what a serious version of §4's
approximator would require.

### Further reading

- Sutton & Barto, *Reinforcement Learning: An Introduction* (2nd ed.), ch. 6 — TD learning and Q-learning. Free online.
- Wagner, "Constructions in combinatorics via neural networks", arXiv:2104.14516 — RL used to refute conjectures.
- Lawson, "Transforming triangulations", *Discrete Mathematics* **3** (1972) — the flip theorem behind §4.
- de Berg et al., *Computational Geometry*, ch. 9 — Delaunay triangulations and the angle-optimality proof.